In [1]:
#STEP 0Setup & load data (run once)

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Load processed EPC data
df = pd.read_csv(
    "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_processed.csv"
)

# Define outcome variable (PEI)
target = "CO2_EMISS_CURR_PER_FLOOR_AREA"

# Drop rows with missing PEI
df = df.dropna(subset=[target])

df.shape

(136, 118)

In [2]:
# ==========================
# Continuous variables correlation with ENERGY_CONSUMPTION_CURRENT
# ==========================

from scipy.stats import pearsonr, spearmanr

# List of continuous variables to test
continuous_vars = [
    "TOTAL_FLOOR_AREA",
    "3_YR_ENERGY_COST_CURRENT",
    "3_YR_ENERGY_SAVINGS_POTENTIAL",
    "ENERGY_CONSUMPTION_CURRENT",
    "SPACE_HEATING_DEMAND",
    "WATER_HEATING_DEMAND",
    "GLAZED_AREA",
    "NUMBER_HABITABLE_ROOMS",
    "NUMBER_HEATED_ROOMS",
    "HEATING_COST_CURRENT",
    "HEATING_COST_POTENTIAL",
    "HOT_WATER_COST_CURRENT",
    "HOT_WATER_COST_POTENTIAL",
    "LIGHTING_COST_CURRENT",
    "LIGHTING_COST_POTENTIAL",
    "EXTENSION_COUNT",
    "LOW_ENERGY_FIXED_LIGHT_COUNT"
]

results_continuous = []

for col in continuous_vars:
    if col in df.columns:
        # Drop missing values for this column
        temp = df[[col, target]].dropna()
        # Skip if column is constant
        if temp[col].nunique() > 1:
            pearson_r, pearson_p = pearsonr(temp[col], temp[target])
            spearman_r, spearman_p = spearmanr(temp[col], temp[target])
            results_continuous.append({
                "Variable": col,
                "Pearson_r": pearson_r,
                "Pearson_p": pearson_p,
                "Spearman_r": spearman_r,
                "Spearman_p": spearman_p
            })

# Convert to DataFrame
df_continuous_corr = pd.DataFrame(results_continuous).sort_values("Spearman_r", key=abs, ascending=False)
df_continuous_corr

,Variable,Pearson_r,Pearson_p,Spearman_r,Spearman_p
3,ENERGY_CONSUMPTION_CURRENT,0.758721,1.028547e-26,0.724357,2.148101e-23
2,3_YR_ENERGY_SAVINGS_POTENTIAL,0.504558,3.772912e-10,0.586755,6.071599e-14
9,HEATING_COST_CURRENT,0.193044,2.434095e-02,0.299955,3.884440e-04
16,LOW_ENERGY_FIXED_LIGHT_COUNT,-0.290307,6.070836e-04,-0.295301,4.827506e-04
1,3_YR_ENERGY_COST_CURRENT,0.198109,2.077957e-02,0.293472,5.252676e-04
12,HOT_WATER_COST_POTENTIAL,-0.298624,4.135013e-04,-0.273504,1.273922e-03
4,SPACE_HEATING_DEMAND,0.111789,1.950765e-01,0.220045,1.005064e-02
7,NUMBER_HABITABLE_ROOMS,0.161478,6.036697e-02,0.219453,1.025854e-02
8,NUMBER_HEATED_ROOMS,0.140732,1.022173e-01,0.195127,2.281771e-02
5,WATER_HEATING_DEMAND,0.197221,2.136931e-02,0.187567,2.876834e-02


In [3]:
# ==========================
# Binary variables correlation with ENERGY_CONSUMPTION_CURRENT
# ==========================

from scipy.stats import pointbiserialr

binary_vars = [
    "MAINS_GAS_FLAG",
    "SOLAR_WATER_HEATING_FLAG",
    "FLAT_TOP_STOREY",
    "PHOTO_SUPPLY"
]

results_binary = []

for col in binary_vars:
    if col in df.columns:
        temp = df[[col, target]].dropna()
        # Try to convert to numeric (0/1)
        try:
            temp[col] = temp[col].astype(int)
        except ValueError:
            # If it can't be converted, skip this column
            print(f"Skipping {col}: not numeric or not binary")
            continue
        
        # Only compute correlation if there is more than one unique value
        if temp[col].nunique() > 1:
            r, p = pointbiserialr(temp[col], temp[target])
            results_binary.append({
                "Variable": col,
                "PointBiserial_r": r,
                "p_value": p
            })

df_binary_corr = pd.DataFrame(results_binary).sort_values("PointBiserial_r", key=abs, ascending=False)
df_binary_corr

Skipping PHOTO_SUPPLY: not numeric or not binary


,Variable,PointBiserial_r,p_value
1,FLAT_TOP_STOREY,-0.485596,0.185108
0,SOLAR_WATER_HEATING_FLAG,-0.066695,0.452670


In [4]:
# ==========================
# Categorical variables correlation with ENERGY_CONSUMPTION_CURRENT
# ==========================

from scipy.stats import f_oneway

categorical_vars = [
    "CONSTRUCTION_AGE_BAND",
    "PROPERTY_TYPE",
    "BUILT_FORM",
    "MAIN_HEATING_CATEGORY",
    "MAIN_FUEL",
    "MAIN_HEATING_CONTROLS",
    "MECHANICAL_VENTILATION",
    "ENERGY_TARIFF",
    "TENURE",
    "TRANSACTION_TYPE",
    "DATA_ZONE",
    "DATA_ZONE_2011",
    "LOCAL_AUTHORITY_LABEL",
    "CONSTITUENCY_LABEL"
]

results_categorical = []

for col in categorical_vars:
    if col in df.columns:
        temp = df[[col, target]].dropna()
        groups = [grp[target].values for name, grp in temp.groupby(col)]
        if len(groups) > 1:
            f_stat, p_val = f_oneway(*groups)
            # Eta-squared as effect size
            ss_between = sum(len(g)*(g.mean() - temp[target].mean())**2 for g in groups)
            eta_squared = ss_between / sum((temp[target] - temp[target].mean())**2)
            results_categorical.append({
                "Variable": col,
                "F_stat": f_stat,
                "p_value": p_val,
                "Eta_squared": eta_squared
            })

df_categorical_corr = pd.DataFrame(results_categorical).sort_values("Eta_squared", ascending=False)
df_categorical_corr

,Variable,F_stat,p_value,Eta_squared
5,MAIN_HEATING_CONTROLS,6.164569,1.766721e-11,0.582594
3,MAIN_HEATING_CATEGORY,19.509306,6.201404e-16,0.485596
4,MAIN_FUEL,6.034313,3.829693e-08,0.380290
9,TRANSACTION_TYPE,4.711214,9.898583e-05,0.204863
0,CONSTRUCTION_AGE_BAND,3.198825,1.698561e-03,0.197473
1,PROPERTY_TYPE,4.637991,4.072498e-03,0.095357
6,MECHANICAL_VENTILATION,4.790254,9.821443e-03,0.068150
8,TENURE,3.028150,3.180426e-02,0.064390
7,ENERGY_TARIFF,2.797942,4.289092e-02,0.062925
11,DATA_ZONE_2011,4.134438,4.399349e-02,0.029931


Here’s a structured correlation report for **current carbon intensity** (`CO2_EMISS_CURR_PER_FLOOR_AREA`), formatted exactly like the PEI report you liked. I’ve included analysis, key variables, and archetyping recommendations.

---

## **1. Continuous Variables**

**Key results:**

| Variable                                                              | Pearson_r       | Pearson_p | Spearman_r | Spearman_p | Notes                                                                                      |
| --------------------------------------------------------------------- | --------------- | --------- | ---------- | ---------- | ------------------------------------------------------------------------------------------ |
| ENERGY_CONSUMPTION_CURRENT                                            | 0.759           | 1e-26     | 0.724      | 2e-23      | Very strong positive correlation; carbon intensity closely tracks energy intensity         |
| 3_YR_ENERGY_SAVINGS_POTENTIAL                                         | 0.505           | 4e-10     | 0.587      | 6e-14      | Moderate to strong correlation; important for retrofit potential                           |
| HEATING_COST_CURRENT                                                  | 0.193           | 0.024     | 0.300      | 0.0004     | Moderate correlation; heating cost contributes to carbon emissions                         |
| 3_YR_ENERGY_COST_CURRENT                                              | 0.198           | 0.021     | 0.293      | 0.0005     | Mirrors heating cost correlation; expected as carbon is largely driven by fuel consumption |
| LOW_ENERGY_FIXED_LIGHT_COUNT                                          | -0.290          | 0.0006    | -0.295     | 0.0005     | Moderate negative correlation; efficient lighting helps reduce carbon intensity            |
| HOT_WATER_COST_POTENTIAL                                              | -0.299          | 0.0004    | -0.274     | 0.0013     | Weak-to-moderate negative correlation; reducing water heating load can help emissions      |
| WATER_HEATING_DEMAND                                                  | 0.197           | 0.021     | 0.188      | 0.029      | Small positive correlation; expected as water heating contributes to carbon                |
| NUMBER_HABITABLE_ROOMS                                                | 0.161           | 0.06      | 0.219      | 0.01       | Slight correlation; larger households slightly higher carbon                               |
| NUMBER_HEATED_ROOMS                                                   | 0.141           | 0.10      | 0.195      | 0.023      | Weak correlation; partially overlaps with habitable rooms                                  |
| SPACE_HEATING_DEMAND                                                  | 0.112           | 0.20      | 0.220      | 0.01       | Weak but possibly non-linear; could inform archetypes visually                             |
| Other variables (TOTAL_FLOOR_AREA, GLAZED_AREA, lighting costs, etc.) | low correlation |           |            |            | Individually weak predictors                                                               |

**Analysis:**

* **ENERGY_CONSUMPTION_CURRENT** is the strongest continuous predictor of carbon intensity — as expected, since most carbon emissions derive from energy use.
* **3_YR_ENERGY_SAVINGS_POTENTIAL** is meaningful for identifying homes where retrofits could significantly reduce carbon.
* **Heating and water costs / demand** are moderate predictors — consistent with major contributors to emissions.
* **Lighting modernization (LOW_ENERGY_FIXED_LIGHT_COUNT)** shows a small but notable effect on emissions.

**Takeaway for archetyping:** Use **ENERGY_CONSUMPTION_CURRENT** as the primary continuous variable. Include **retrofit potential** and **heating/water load** if subtypes are desired.

---

## **2. Binary Variables**

| Variable                 | PointBiserial_r | p_value | Notes                                                            |
| ------------------------ | --------------- | ------- | ---------------------------------------------------------------- |
| FLAT_TOP_STOREY          | -0.486          | 0.185   | Moderate negative correlation, but not statistically significant |
| SOLAR_WATER_HEATING_FLAG | -0.067          | 0.453   | Very weak; negligible for archetyping                            |

**Analysis:**

* Binary features are **not strongly correlated** with carbon intensity.
* They could serve as **descriptive attributes** (e.g., noting which dwellings have solar water heating) but are unlikely to differentiate archetypes statistically.

**Takeaway for archetyping:** Focus on continuous and categorical variables; use binary flags for description only.

---

## **3. Categorical Variables (ANOVA / Eta²)**

| Variable                      | F_stat  | p_value  | Eta²  | Notes                                                                       |
| ----------------------------- | ------- | -------- | ----- | --------------------------------------------------------------------------- |
| MAIN_HEATING_CONTROLS         | 6.16    | 1.77e-11 | 0.583 | Very strong effect; controls type strongly influences carbon intensity      |
| MAIN_HEATING_CATEGORY         | 19.51   | 6.2e-16  | 0.486 | Strong categorical predictor; heating system type central to carbon profile |
| MAIN_FUEL                     | 6.03    | 3.83e-08 | 0.380 | Significant effect; electric vs gas vs oil has large carbon impact          |
| TRANSACTION_TYPE              | 4.71    | 9.9e-05  | 0.205 | Moderate effect; occupancy/usage patterns may influence carbon emissions    |
| CONSTRUCTION_AGE_BAND         | 3.20    | 0.0017   | 0.197 | Older buildings tend to have higher emissions, but retrofits can confound   |
| PROPERTY_TYPE                 | 4.64    | 0.004    | 0.095 | Minor effect; detached vs semi-detached differences                         |
| MECHANICAL_VENTILATION        | 4.79    | 0.0098   | 0.068 | Small but significant contributor                                           |
| TENURE                        | 3.03    | 0.032    | 0.064 | Small effect; occupancy pattern may influence emissions                     |
| ENERGY_TARIFF                 | 2.80    | 0.043    | 0.063 | Small effect                                                                |
| DATA_ZONE_2011                | 4.13    | 0.044    | 0.030 | Minor contribution                                                          |
| Other (DATA_ZONE, BUILT_FORM) | not sig | >0.25    | <0.01 | Negligible                                                                  |

**Analysis:**

* **MAIN_HEATING_CATEGORY, MAIN_HEATING_CONTROLS, and MAIN_FUEL** are the most significant categorical drivers of carbon intensity.
* Occupancy, tenure, and minor building characteristics contribute modestly but are less critical.

**Takeaway for archetyping:** Use **heating system type, controls, and fuel** as key categorical axes.

---

## **4. Recommended Variables for Carbon Archetyping**

**Primary (strong predictors, statistically significant):**

1. **ENERGY_CONSUMPTION_CURRENT** – continuous, main driver of carbon intensity
2. **MAIN_HEATING_CATEGORY** – categorical, heating type is central
3. **MAIN_HEATING_CONTROLS** – categorical, influences actual usage
4. **MAIN_FUEL** – categorical, significant impact on carbon

**Secondary (descriptive / moderate predictors):**

* 3_YR_ENERGY_SAVINGS_POTENTIAL
* Heating/water costs or demand
* Binary flags (solar water heating, flat top storey) as descriptive labels

---

## **5. Suggested Procedure for Carbon Archetyping**

1. **Step 1 – Select variables:**

   * Continuous: `ENERGY_CONSUMPTION_CURRENT` (+ optional 3_YR_ENERGY_SAVINGS_POTENTIAL)
   * Categorical: `MAIN_HEATING_CATEGORY`, `MAIN_HEATING_CONTROLS`, `MAIN_FUEL`

2. **Step 2 – Initial grouping:**

   * For categorical variables, define archetypes by combinations (e.g., “Electric storage heaters + basic controls + gas fuel”).
   * For continuous variables, bin into tertiles or quartiles to separate low/medium/high carbon intensity.

3. **Step 3 – Statistical verification:**

   * Compute within-group variance for continuous variables to ensure homogeneity.
   * Check between-group variance to confirm categories are well-separated.

4. **Step 4 – Align with local knowledge:**

   * Compare archetypes with actual Fintry dwellings for interpretability.
   * Adjust bins or categories for intuitive archetype naming while retaining statistical separation.

5. **Step 5 – Use archetypes in modeling:**

   * Assign each dwelling to a carbon archetype.
   * Run PHPP / hourly models per archetype.
   * Store representative carbon/emission profiles for the library.

---

✅ **Summary:**

* **Most predictive variables:** ENERGY_CONSUMPTION_CURRENT, MAIN_HEATING_CATEGORY, MAIN_HEATING_CONTROLS, MAIN_FUEL
* **Next steps:** Focus carbon archetypes on heating system type, controls, fuel, and energy consumption intensity.
* **Binary/minor categorical variables** are descriptive only, not defining features.

---

If you want, I can **also produce a combined PEI + Carbon + Cost correlation summary table**, so you can prioritize archetypes along all three metrics at once — would make the decision-making much faster. Do you want me to do that next?
